<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/chatterbox-finetuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Chatterbox TTS Fine-Tuning on Google Colab

This notebook provides a complete workflow for fine-tuning Chatterbox TTS models (Standard and Turbo) with LoRA support.

## Overview
- **Step 1**: Clone repository
- **Step 2**: Connect Google Drive and Copy Dataset
- **Step 3**: Install Dependencies
- **Step 4**: Configure Training Parameters (Select LoRA/Full Fine-tune, Turbo/Non-Turbo)
- **Step 5**: Download & Prepare Models (setup.py)
- **Step 6**: Preprocess & Train Model (Combined - depends on config.py settings from Step 4)
- **Step 7**: Inference (Generate Speech)
- **Step 8**: Merge LoRA Adapter (Optional)


## Step 1: Clone Repository

In [ ]:
#@title 📥 Clone Chatterbox Repository

import os

# Detect environment and set working directory
if os.path.exists('/content'):
    # Running in Google Colab
    WORKSPACE = '/content'
    %cd /content
    print("Detected Google Colab environment")
else:
    # Running locally or in other environments
    WORKSPACE = '/workspace'
    %cd "$WORKSPACE"
    print("Running in local/other environment")

# Clone the repository if it doesn't exist
if not os.path.exists('chatterbox-finetuning'):
    print("Cloning Chatterbox Finetuning repository...")
    !git clone https://github.com/Amazeble/chatterbox-finetuning.git
    %cd chatterbox-finetuning
    print("✅ Repository cloned successfully!")
else:
    print("✅ Repository already exists.")
    %cd chatterbox-finetuning

# Verify contents
print("\nRepository contents:")
!ls -la

## Step 2: Connect Google Drive and Copy Dataset

In [ ]:
#@title 📁 Connect Google Drive and Copy Dataset

import os
from google.colab import drive

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define source and destination paths
source_path = '/content/drive/MyDrive/MyTTSDataset/'
dest_path = '/content/chatterbox-finetuning/'

# Check if source path exists
if os.path.exists(source_path):
    print(f"\n✅ Source path found: {source_path}")
    print("Copying dataset to working directory...")

    # Create destination directory if it doesn't exist
    os.makedirs(dest_path, exist_ok=True)

    # Copy files
    !cp -r {source_path}/* {dest_path}

    print(f"✅ Dataset copied successfully to {dest_path}")
    print("\nVerifying copied files:")
    !ls -la {dest_path}
else:
    print(f"⚠️ Source path not found: {source_path}")
    print("Please ensure your dataset is in the correct location in Google Drive.")

## Step 3: Install Dependencies

In [ ]:
#@title 🔧 Install Dependencies

import os

# Install FFmpeg
print("Installing FFmpeg...")
!apt-get update && apt-get install -y ffmpeg

# Install Python dependencies
print("\nInstalling Python dependencies...")
%cd /content/chatterbox-finetuning
!pip install -r requirements.txt

print("\n✅ Dependencies installed successfully!")

## Step 4: Configure Training Parameters

In [ ]:
#@title ⚙️ Configure Training Parameters

import os

# Check if repository exists
repo_path = '/content/chatterbox-finetuning'
if not os.path.exists(repo_path):
    raise FileNotFoundError(f"❌ Repository not found at {repo_path}! Please run Step 1 (Clone Repository) first.")

%cd /content/chatterbox-finetuning

# Interactive Configuration Widgets
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

print("="*50)
print("🎛️ Training Configuration")
print("="*50)

# Read current config values
with open('src/config.py', 'r') as f:
    config_content = f.read()

# Parse current values from config
def get_config_value(pattern, default):
    match = re.search(pattern, config_content)
    if match:
        value = match.group(1)
        if value.lower() == 'true':
            return True
        elif value.lower() == 'false':
            return False
        try:
            return int(value)
        except:
            return default
    return default

# Get current values
current_turbo = get_config_value(r'is_turbo:\s*bool\s*=\s*(\w+)', True)
current_lora = get_config_value(r'is_lora:\s*bool\s*=\s*(\w+)', True)
current_batch = get_config_value(r'batch_size:\s*int\s*=\s*(\d+)', 8)
current_lr_match = re.search(r'learning_rate:\s*float\s*=\s*([\d.e+-]+)', config_content)
current_lr = float(current_lr_match.group(1)) if current_lr_match else 1e-4
current_epochs = get_config_value(r'num_epochs:\s*int\s*=\s*(\d+)', 10)
current_ljspeech = get_config_value(r'ljspeech\s*=\s*(\w+)', True)
current_json = get_config_value(r'json_format\s*=\s*(\w+)', False)
# Get preprocess value - could be True, False, or 'auto'
preprocess_match = re.search(r"preprocess\s*=\s*(['"]?)(\w+)\1", config_content)
if preprocess_match:
    current_preprocess_raw = preprocess_match.group(2)
else:
    current_preprocess_raw = 'True'
current_vocab = get_config_value(r'new_vocab_size:\s*int\s*=\s*(\d+)', 52260)

# Create widgets for key settings
is_turbo_widget = widgets.Checkbox(
    value=current_turbo,
    description='Turbo Mode',
    disabled=False,
    tooltip='Enable Turbo mode for faster training'
)

is_lora_widget = widgets.Checkbox(
    value=current_lora,
    description='LoRA Training',
    disabled=False,
    tooltip='Use LoRA (recommended for <10h data). Uncheck for Full Fine-tune'
)

ljspeech_widget = widgets.Checkbox(
    value=current_ljspeech,
    description='LJSpeech Format',
    disabled=False,
    tooltip='Set True if dataset format is LJSpeech (ID|RawText|NormText)'
)

json_format_widget = widgets.Checkbox(
    value=current_json,
    description='JSON Format',
    disabled=False,
    tooltip='Set True if dataset format is JSON'
)

preprocess_widget = widgets.Checkbox(
    value=current_preprocess,
    description='Run Preprocessing',
    disabled=False,
    tooltip='Set True for first run to preprocess dataset'
)

batch_size_widget = widgets.IntSlider(
    value=current_batch,
    min=1,
    max=16,
    step=1,
    description='Batch Size',
    disabled=False,
    tooltip='Adjust based on VRAM (2, 4, 8 recommended)'
)

learning_rate_widget = widgets.FloatLogSlider(
    value=current_lr,
    base=10,
    min=-5,
    max=-3,
    step=0.1,
    description='Learning Rate',
    disabled=False,
    tooltip='Lower for full fine-tune (1e-5), higher for LoRA (1e-4)'
)

num_epochs_widget = widgets.IntSlider(
    value=current_epochs,
    min=1,
    max=50,
    step=1,
    description='Epochs',
    disabled=False,
    tooltip='Number of training epochs'
)

vocab_size_widget = widgets.IntText(
    value=current_vocab,
    description='Vocab Size',
    disabled=False,
    tooltip='Must match tokenizer.json (update after setup.py if Turbo)'
)

display(Markdown("**Training Options:**"))
display(is_turbo_widget)
display(is_lora_widget)
display(Markdown("**Dataset Format:**"))
display(ljspeech_widget)
display(json_format_widget)
display(preprocess_widget)
display(Markdown("**Hyperparameters:**"))
display(batch_size_widget)
display(learning_rate_widget)
display(num_epochs_widget)
display(vocab_size_widget)

print("\n" + "="*50)
print("Click the button below to apply your settings to config.py")
print("="*50)

# Create apply button
apply_button = widgets.Button(
    description='Apply Settings to config.py',
    button_style='success',
    tooltip='Click to apply the widget settings to src/config.py'
)

output = widgets.Output()

def on_apply_clicked(b):
    with output:
        clear_output()
        # Get widget values
        turbo_value = "True" if is_turbo_widget.value else "False"
        lora_value = "True" if is_lora_widget.value else "False"
        ljspeech_value = "True" if ljspeech_widget.value else "False"
        json_value = "True" if json_format_widget.value else "False"
        preprocess_value = "True" if preprocess_widget.value else "False"
        batch_value = str(batch_size_widget.value)
        lr_value = str(learning_rate_widget.value)
        epochs_value = str(num_epochs_widget.value)
        vocab_value = str(vocab_size_widget.value)

        # Read current config
        with open('src/config.py', 'r') as f:
            config_content = f.read()

        # Update all settings
        config_content = re.sub(r'is_turbo:\s*bool\s*=\s*.*', f'is_turbo: bool = {turbo_value}', config_content)
        config_content = re.sub(r'is_lora:\s*bool\s*=\s*.*', f'is_lora: bool = {lora_value}', config_content)
        config_content = re.sub(r'ljspeech\s*=\s*.*', f'ljspeech = {ljspeech_value}', config_content)
        config_content = re.sub(r'json_format\s*=\s*.*', f'json_format = {json_value}', config_content)
        config_content = re.sub(r'preprocess\s*=\s*.*', f'preprocess = {preprocess_value}', config_content)
        config_content = re.sub(r'batch_size:\s*int\s*=\s*.*', f'batch_size: int = {batch_value}', config_content)
        config_content = re.sub(r'learning_rate:\s*float\s*=\s*.*', f'learning_rate: float = {lr_value}', config_content)
        config_content = re.sub(r'num_epochs:\s*int\s*=\s*.*', f'num_epochs: int = {epochs_value}', config_content)
        config_content = re.sub(r'new_vocab_size:\s*int\s*=\s*.*', f'new_vocab_size: int = {vocab_value}', config_content)

        # Write updated config
        with open('src/config.py', 'w') as f:
            f.write(config_content)

        print(f"✅ Applied settings to src/config.py:")
        print(f"  - is_turbo={turbo_value}")
        print(f"  - is_lora={lora_value}")
        print(f"  - ljspeech={ljspeech_value}")
        print(f"  - json_format={json_value}")
        print(f"  - preprocess={preprocess_value}")
        print(f"  - batch_size={batch_value}")
        print(f"  - learning_rate={lr_value}")
        print(f"  - num_epochs={epochs_value}")
        print(f"  - new_vocab_size={vocab_value}")
        print("\n⚠️ IMPORTANT: Now proceed to Step 5 to run setup.py!")
        print("After setup.py completes, if using Turbo mode, check the output for vocab_size.")
        print("You may need to update new_vocab_size above and click Apply again.")

apply_button.on_click(on_apply_clicked)
display(apply_button)
display(output)

# Show current config
print("\n" + "="*50)
print("Current configuration (src/config.py):")
print("="*50)
!cat src/config.py


In [ ]:
#@title 📥 Download & Prepare Models (setup.py)

%cd /content/chatterbox-finetuning

print("Running setup.py to download models...")
print("This may take several minutes depending on your internet connection.")

# Run setup.py
!python setup.py

print("\n✅ Setup completed!")
print("Check the output above for the vocab_size if using Turbo mode.")
print("You may need to update new_vocab_size in src/config.py with the value from setup.py output.")

In [ ]:
#@title 🔄 Preprocess & Train Model
#@markdown This combined cell handles both preprocessing and training based on your config.py settings.
#@markdown **Preprocessing** depends on the settings from Step 4 (config.py):
#@markdown - `preprocess=True`: Will preprocess dataset before training
#@markdown - `preprocess=False`: Will skip preprocessing and go straight to training
#@markdown
#@markdown **Training** will use these settings from config.py:
#@markdown - `is_turbo`: Turbo mode for faster training
#@markdown - `is_lora`: LoRA fine-tuning (recommended for <10h data)
#@markdown - `batch_size`, `learning_rate`, `num_epochs`: Training hyperparameters

%cd /content/chatterbox-finetuning

print("="*50)
print("🔄 Preprocess & Train Model")
print("="*50)

# Read config to show current settings
with open('src/config.py', 'r') as f:
    config_content = f.read()

print("\n📋 Current Configuration:")
print("-" * 50)

# Extract key settings
import re
settings = {
    'Turbo Mode': r'is_turbo:\s*bool\s*=\s*(\w+)',
    'LoRA Training': r'is_lora:\s*bool\s*=\s*(\w+)',
    'Batch Size': r'batch_size:\s*int\s*=\s*(\d+)',
    'Learning Rate': r'learning_rate:\s*float\s*=\s*([\d.e+-]+)',
    'Epochs': r'num_epochs:\s*int\s*=\s*(\d+)',
    'Preprocess': r'preprocess\s*=\s*(\w+)',
    'LJSpeech Format': r'ljspeech\s*=\s*(\w+)',
    'JSON Format': r'json_format\s*=\s*(\w+)'
}

for name, pattern in settings.items():
    match = re.search(pattern, config_content)
    if match:
        print(f"{name}: {match.group(1)}")

print("-" * 50)

if 'preprocess=True' in config_content or 'preprocess = True' in config_content:
    print("\n⏳ Starting preprocessing (this may take several minutes)...")
    !python train.py
    print("\n✅ Preprocessing completed!")
    print("\n" + "="*50)
    print("🏋️ Starting training...")
    print("="*50)
    print("Training progress and audio samples will be displayed below.")
    print("Press Ctrl+C to stop training early.")
    !python train.py
else:
    print("\n⏳ Skipping preprocessing (preprocess=False in config.py)")
    print("\n🏋️ Starting training...")
    print("="*50)
    print("Training progress and audio samples will be displayed below.")
    print("Press Ctrl+C to stop training early.")
    !python train.py

print("\n✅ Training completed!")
print("Check the chatterbox_output/ directory for your trained model.")

In [ ]:
#@title 🗣️ Inference - Generate Speech

%cd /content/chatterbox-finetuning

# First, let's check what models are available
print("Checking available models...")
!ls -la chatterbox_output/

print("\n" + "="*50)
print("To generate speech, edit inference.py with your desired text:")
print("  TEXT_TO_SAY = \"Your text here\"")
print("  AUDIO_PROMPT = \"./speaker_reference/reference.wav\"")
print("="*50)

# Run inference
print("\nRunning inference...")
!python inference.py

print("\n✅ Speech generated!")
print("Output saved as output.wav")

# Play the generated audio
from IPython.display import Audio, display
print("\nPlaying generated audio:")
display(Audio('./output.wav'))

In [ ]:
#@title 📦 Merge LoRA Adapter (Optional)

%cd /content/chatterbox-finetuning

print("Merging LoRA adapter into base model...")
print("This creates a standalone .safetensors file that doesn't require LoRA adapters.")

!python merge_lora.py

print("\n✅ Merge completed!")
print("Your merged model is ready for production use.")
print("Check chatterbox_output/ for the merged .safetensors file.")